# Modélisation vol + météo

## 1. Contexte et objectif du notebook

Les trois sources sont déjà dans PostgreSQL (schéma en étoile). Je m'en sers ici pour un **premier modèle** : vols, météo, et actualités jointes **par date UTC de `first_seen`**.

Le score BERT (`dim_news_sentiment.score_sentiment`) est produit par `src/processing/scorer_sentiment.py` et documenté dans `notebooks/05_sentiment_bert.ipynb`. Ce notebook-ci reste le diagnostic cible / split / LightGBM. Comparaison : `python -m src.models.entrainer` (avec actu) et `--sans-actu`.

In [1]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd

RACINE_PROJET = os.path.abspath(os.path.join(os.getcwd(), ".."))
if RACINE_PROJET not in sys.path:
    sys.path.insert(0, RACINE_PROJET)
os.chdir(RACINE_PROJET)

from src.models.entrainer import executer_pipeline
from src.models.jeu import (
    COLONNES_FEATURES,
    COLONNES_INTERDITES_FEATURES,
    NOM_CIBLE,
    ajouter_cible_duree,
    charger_vols_meteo,
    construire_features,
    diagnostiquer_volumetrie,
    exclure_durees_aberrantes,
    filtrer_jeu_modele,
    retenir_cible,
    split_temporel,
)

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", message="Pandas requires version")

C:\Users\jeann\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 2. Chargement depuis PostgreSQL

Je lis `fact_flights` joint à `dim_date`, `dim_weather` et `dim_news_sentiment` (date UTC de `first_seen`) via `charger_vols_meteo`. Aucun appel HTTP. Si l'instance Postgres n'est pas joignable, la fonction échoue.

In [2]:
df = charger_vols_meteo()
print(f"Lignes : {len(df)}")
print(f"Colonnes : {len(df.columns)}")
print(f"first_seen min / max : {df['first_seen'].min()} / {df['first_seen'].max()}")
df.head(3)

2026-08-28 14:28:57,400 - src.models.jeu - INFO - Jeu brut chargé : 1286 ligne(s), 24 colonne(s).


Lignes : 1286
Colonnes : 24
first_seen min / max : 2026-08-25 13:20:36+00:00 / 2026-08-28 12:25:15+00:00


,flight_id,icao24,callsign,first_seen,last_seen,est_departure_airport,est_arrival_airport,date_id,retard_minutes,weather_origin_id,weather_dest_id,jour_semaine,mois,temperature_origine,vent_origine,rafale_origine,pluie_origine,visibilite_origine,humidite_origine,temperature_dest,vent_dest,rafale_dest,pluie_dest,visibilite_dest
0,753,344416,IBS1527,2026-08-25 19:33:29+00:00,2026-08-25 21:43:31+00:00,LEMD,None,20260825,NaN,NaN,NaN,2,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,754,4ca2bf,RYR1GR,2026-08-25 19:21:15+00:00,2026-08-25 21:10:28+00:00,LEMD,LIAU,20260825,NaN,NaN,NaN,2,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,755,344341,AEA9172,2026-08-25 19:20:20+00:00,2026-08-25 21:41:47+00:00,LEMD,None,20260825,NaN,NaN,NaN,2,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Volumétrie, météo, proxy de retard

Je passe le DataFrame à `diagnostiquer_volumetrie` : totaux, part de météo origine / destination, NULL sur `retard_minutes`, routes assez peuplées pour un historique.

In [3]:
diag = diagnostiquer_volumetrie(df)
for cle, valeur in diag.items():
    print(f"{cle}: {valeur}")

nb_vols: 1286
retard_non_null: 11
pct_retard_null: 99.1
weather_origin: 711
pct_weather_origin: 55.3
weather_dest: 12
pct_weather_dest: 0.9
dest_renseignee: 308
last_seen_ok: 1286
meteo_et_duree: 711
routes_ge5: 3
vols_sur_routes_ge5: 17
routes_meteo_ge5: 0
vols_meteo_sur_routes_ge5: 0
first_seen_min: 2026-08-25T13:20:36+00:00
first_seen_max: 2026-08-28T12:25:15+00:00
nb_jours: 4


**Observations** : les cellules ci-dessus peuvent encore afficher un instantané antérieur (ex. 1286 / 1290 vols) si le notebook n'a pas été relancé. Référence CLI du 01/09/2026 13:19 : **7524** vols, **100** météo, météo origine **1335** (17,7 %), destination **20** (0,3 %), dest ICAO **5564** (74,0 %), proxy `retard_minutes` **1663** dont **4** avec météo origine (77,9 % de NULL). Fenêtre `first_seen` : 25/08–01/09/2026 UTC (7 jours calendaires, pas de vol le 29/08). `dim_news_sentiment` : **26** jours (27/07–31/08), `score_sentiment` **tout NULL**. Recouvrement : **6 jours** communs, **6821 / 7524 = 90,7 %** des vols avec `nb_articles_actu` > 0 ; le **01/09** (703 vols, tout le test) sans article. Passe NewsAPI 13:18 : **5 requêtes**, +7 jours. Statut du Rendu 3 : brouillon.

## 4. Choix de cible

Trois options étaient possibles :

- C : entraîner le proxy `retard_minutes` sur les lignes non NULL (1663 lignes, 4 avec météo origine).
- A : classer « plus long que la médiane historique du couple OD » s'il y a assez de vols **par route**.
- B : régresser la **durée observée** `last_seen − first_seen`.

`retenir_cible` applique un seuil de 50 lignes.

In [4]:
code_cible = retenir_cible(diag)
print(f"Cible retenue : {code_cible}")
print(f"Nom de colonne : {NOM_CIBLE}")
print(f"retard_minutes non NULL : {diag['retard_non_null']}")
print(f"retard_minutes non NULL avec météo origine : {diag['retard_non_null_avec_meteo']}")
print(f"vols sur routes OD >= 5 avec météo origine : {diag['vols_meteo_sur_routes_ge5']}")
print(f"vols avec météo origine et durée : {diag['meteo_et_duree']}")

Cible retenue : B
Nom de colonne : duree_minutes
retard_minutes non NULL : 11
vols sur routes OD >= 5 avec météo origine : 0
vols avec météo origine et durée : 711


**Observations** : le diagnostic retient **B**. C reste sous le seuil : 1663 lignes de proxy, **4** avec météo origine (il en faudrait 50 pour basculer). A n'a aucune route assez peuplée après filtre météo (256 routes ≥ 5 vols hors filtre, 0 parmi les 1335 avec météo origine). B a **1335** durées avec météo origine, **1329** après le plafond 8 h (6 écartées). Ce n'est plus « prédiction de retard » au sens DGAC (réel − programmé) : c'est une régression de durée ADS-B, défendable parce que je ne fabrique pas le slot manquant.

## 5. Features et fuite de données

Je filtre les lignes sans météo origine, je calcule `duree_minutes`, j'écarte les durées > 8 h, puis `construire_features` assemble X (météo +, par défaut, `nb_articles_actu` et `proxy_perturbation_actu`). `last_seen` sert à la cible ; il ne doit pas se retrouver dans X. Les actualités se joignent par date UTC de `first_seen`, pas par vol.

In [5]:
jeu = filtrer_jeu_modele(df)
jeu = ajouter_cible_duree(jeu)
jeu = exclure_durees_aberrantes(jeu)
print(f"Lignes après filtre météo origine et plafond 8 h : {len(jeu)}")
print(jeu[NOM_CIBLE].describe().round(2))

X, y = construire_features(jeu)
print("Colonnes de X :", list(X.columns))
print("Intersection avec colonnes interdites :", set(X.columns) & COLONNES_INTERDITES_FEATURES)
print("last_seen dans X :", "last_seen" in X.columns)
print("cible dans X :", NOM_CIBLE in X.columns)
X.head(3)

2026-08-28 14:28:57,510 - src.models.jeu - INFO - Jeu modèle : 711 ligne(s) avec météo origine et durée observée.


Lignes après filtre météo origine : 711
count     711.00
mean       69.85
std        80.97
min         0.32
25%        29.64
50%        53.97
75%        89.97
max      1412.65
Name: duree_minutes, dtype: float64
Colonnes de X : ['hub_origine', 'jour_semaine', 'mois', 'heure_utc', 'temperature_origine', 'vent_origine', 'rafale_origine', 'pluie_origine', 'visibilite_origine', 'humidite_origine', 'temperature_dest', 'vent_dest', 'rafale_dest', 'pluie_dest', 'visibilite_dest']
Intersection avec colonnes interdites : set()
last_seen dans X : False
cible dans X : False


,hub_origine,jour_semaine,mois,heure_utc,temperature_origine,vent_origine,rafale_origine,pluie_origine,visibilite_origine,humidite_origine,temperature_dest,vent_dest,rafale_dest,pluie_dest,visibilite_dest
15,LEMD,2,8,17.433333,27.63,6.69,0.0,0.0,10000.0,38.0,NaN,NaN,NaN,NaN,NaN
16,LEMD,2,8,17.116667,27.63,6.69,0.0,0.0,10000.0,38.0,NaN,NaN,NaN,NaN,NaN
17,LEMD,2,8,16.900000,27.63,6.69,0.0,0.0,10000.0,38.0,NaN,NaN,NaN,NaN,NaN


In [6]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(y, bins=40, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Durée observée (minutes)")
ax.set_ylabel("Nombre de vols")
ax.set_title("Distribution de duree_minutes (jeu avec météo origine)")
plt.tight_layout()
plt.show()

C:\Users\jeann\AppData\Local\Temp\ipykernel_13836\2508720148.py:7: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


**Observations** : 1329 lignes après plafond 8 h (6 durées > 480 min écartées parmi les 1335 avec météo origine ; max brut du jeu ~30,5 h, 373 durées > 8 h sur les 7524). Les sorties ci-dessus peuvent encore montrer 710 / 711 si la cellule n'a pas été relancée.

Médiane de durée autour de 50–55 minutes, moyenne plus haute : des durées ADS-B qui ne sont pas de « vrais » temps de vol porte-à-porte.

X contient le hub, le calendrier (jour, mois, heure UTC tirée de `first_seen`), la météo origine, la météo dest souvent vide, et par défaut `nb_articles_actu` / `proxy_perturbation_actu`. Sur le jeu modèle (1329 lignes) : **710** ont `nb_articles_actu` > 0 (53,4 %) — surtout le train ; le test du 01/09 est à 0. Rien de `last_seen`, ni du proxy retard. La destination ICAO n'est pas une feature.

## 6. Split temporel

Je coupe sur `first_seen` au quantile 70 % : tout ce qui est **avant** le seuil va au train, tout ce qui est **après** au test. Pas de tirage aléatoire i.i.d.

In [7]:
train, test = split_temporel(jeu, fraction_train=0.70)
print(f"Train : {len(train)} vols, jusqu'à {train['first_seen'].max()}")
print(f"Test  : {len(test)} vols, depuis {test['first_seen'].min()}")
print(
    "max(train) < min(test) :",
    train["first_seen"].max() < test["first_seen"].min(),
)

2026-08-28 14:28:57,793 - src.models.jeu - INFO - Split temporel : train=497 (jusqu'à 2026-08-26T14:43:31+00:00), test=214 (depuis 2026-08-26T14:43:43+00:00).


Train : 497 vols, jusqu'à 2026-08-26 14:43:31+00:00
Test  : 214 vols, depuis 2026-08-26 14:43:43+00:00
max(train) < min(test) : True


**Observations** : CLI 01/09/2026 13:19 après plafond 8 h — **930** vols en train (jusqu'à 01/09/2026 09:33:06 UTC), **399** en test (depuis 09:33:10 UTC). Même split avec/sans actu (les news ne changent pas le filtre). Train : **710 / 930 = 76,3 %** avec `nb_articles_actu` > 0. Test : **0 / 399** (le 01/09 n'a aucun article). Le test tient dans ~1 h 15 le 01/09 (max 10:47 UTC). Avant la salve OAuth : 497 / 213, seuil 26/08 14:44. Un k-fold mélangerait passé et futur ; je ne l'utilise pas.

## 7. Baseline médiane et LightGBM

Même enchaînement que `python -m src.models.entrainer` : `executer_pipeline` sur le DataFrame déjà chargé (pas de second aller-retour SQL). Deux baselines : médiane globale du train, et médiane **par heure UTC** (repli sur la globale si l'heure a moins de 5 vols en train). Pas de sauvegarde `joblib` ici (le script CLI l'écrit dans `models/`).

In [8]:
rapport = executer_pipeline(jeu=df, sauvegarder=False)
print(f"Cible : {rapport['cible']} (code {rapport['code_cible']})")
print(f"Train / test : {rapport['n_train']} / {rapport['n_test']}")
print(f"Fin train : {rapport['fin_train']}")
print(f"Début test : {rapport['debut_test']}")
print("LightGBM :", {k: round(v, 2) for k, v in rapport['metriques_modele'].items()})
print("Médiane globale :", {k: round(v, 2) for k, v in rapport['metriques_baseline'].items()})
print("Médiane horaire :", {k: round(v, 2) for k, v in rapport['metriques_baseline_horaire'].items()})
print(f"Bat la globale : {rapport['bat_la_baseline']}")
print(f"Bat l'horaire : {rapport['bat_la_baseline_horaire']}")

2026-08-28 14:28:57,830 - src.models.jeu - INFO - Jeu modèle : 711 ligne(s) avec météo origine et durée observée.
2026-08-28 14:28:57,844 - src.models.jeu - INFO - Split temporel : train=497 (jusqu'à 2026-08-26T14:43:31+00:00), test=214 (depuis 2026-08-26T14:43:43+00:00).
2026-08-28 14:28:58,172 - src.models.entrainer - INFO - Cible duree_minutes (B). Train=497, test=214. LightGBM RMSE=74.00 MAE=66.52. Baseline médiane RMSE=43.77 MAE=37.94. Bat la baseline : False.


Cible : duree_minutes (code B)
Train / test : 497 / 214
Fin train : 2026-08-26T14:43:31+00:00
Début test : 2026-08-26T14:43:43+00:00
LightGBM : {'rmse': 74.0, 'mae': 66.52}
Baseline : {'rmse': 43.77, 'mae': 37.94}
Bat la baseline : False


**Observations** : CLI 01/09/2026 13:19–13:20 (7524 vols, 1329 lignes après plafond 8 h, 26 jours d'actu) — LightGBM **avec** actualités RMSE **55,22** / MAE **50,54** ; `--sans-actu` **56,72** / **51,82**. Médiane globale **40,43** / **34,60**, médiane horaire **69,60** / **63,80**. **Bat la globale : False. Bat l'horaire : True**. L'actu aide de −1,5 RMSE, **pas assez** pour battre la médiane. Le test est le 01/09, **sans article** : ce n'est pas « l'actu prédit le test ». `nb_articles_actu` : 50 splits ; proxy lexical : 0. Pas de BERT. Avant la passe NewsAPI (12:52) : avec = sans = 56,72. Avant OAuth (1290 vols) : 80,43 vs 43,88 / 44,48. Un jet trop court le 28/08 (40,20 vs 53,83) n'est pas à citer. Les sorties notebook ci-dessus (74,00 vs 43,77) restent un instantané d'avant le plafond et d'avant la salve 200.

## 8. Quelles variables ont servi ?

In [9]:
importances = pd.Series(rapport["importance_features"]).sort_values(ascending=True)
print(importances.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
importances.plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel("Importance LightGBM (splits)")
ax.set_title("Importance des features")
plt.tight_layout()
plt.show()

mois                     0
rafale_origine           0
pluie_origine            0
visibilite_origine       0
temperature_dest         0
vent_dest                0
rafale_dest              0
pluie_dest               0
visibilite_dest          0
hub_origine              9
jour_semaine            88
humidite_origine       105
temperature_origine    152
vent_origine           170
heure_utc              799


C:\Users\jeann\AppData\Local\Temp\ipykernel_13836\799537175.py:9: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


**Observations** : l'heure UTC domine encore (915 splits), devant la température (155), le vent (125), le jour de semaine (116), le hub (84) et l'humidité (71) à l'origine. `nb_articles_actu` passe à **50** splits (plus une constante). `proxy_perturbation_actu` reste à **0**. Le mois à 23, la pluie origine à 7. Rafales, visibilité, toute la météo destination à zéro. Le modèle ne bat pas la médiane globale (40,43) ; il bat la médiane horaire (69,60), trop faible sur ce créneau du 01/09.

## 9. Limites

- **Pas un retard** : pas d'horaire programmé, proxy inutilisable (1663 lignes, 4 avec météo origine).
- **Volume** : 1329 lignes après plafond 8 h, sept jours de `first_seen`, dix hubs. 17,7 % des vols ont une météo origine (fenêtre ± 3 h vs salve `/flights` 24 h). `dim_weather` : 100 observations.
- **Destination** : 74 % renseignée hors filtre, 0 route OD ≥ 5 vols une fois la météo origine exigée. Les 6 durées > 8 h du jeu modèle (373 sur 7524, max ~30,5 h) sont écartées.
- **Actualités** : jointure par date UTC de `first_seen`. `score_sentiment` tout NULL (pas de BERT). 26 jours en base (27/07–31/08), **6 jours** communs avec les vols, 90,7 % des faits avec `nb_articles_actu` > 0. Le test du 01/09 n'a aucun article. RMSE 55,22 avec actu vs 56,72 sans, vs médiane 40,43 : **ça n'aide toujours pas** à battre la baseline. Je n'invente pas de scores BERT.
- **LightGBM perd** contre la médiane globale (RMSE 55,22 vs 40,43) et **gagne** contre la médiane horaire (69,60). Avant OAuth, 1290 vols : 80,43 vs 43,88 / 44,48. Un jet antérieur (40,20 vs 53,83 sur 972 vols, test trop court) n'est pas le résultat à citer.
- Relancer le notebook après une nouvelle collecte / un nouvel ETL changera les COUNT et donc les métriques. Les chiffres commentés ici s'alignent sur l'entraînement CLI du 01/09/2026 13:19–13:20 (voir Rendu 3, brouillon).

Pour réécrire l'artefact entraîné : `python -m src.models.entrainer` (avec actu) ou `--sans-actu --sans-sauvegarde` (échec explicite si PostgreSQL est injoignable). OpenSky : un smoke `python -m src.ingestion.opensky_client --smoke` avant toute salve. NewsAPI ciblée : `python -m src.ingestion.newsapi_client --depuis 2026-08-25 --jusqua 2026-09-01T23:59:59` (5 requêtes, pas de boucle par jour).